# Enterprise SIEM Lab — Offline Demonstration

**Status:** This is an educational simulation using synthetic events. It is not a live Wazuh deployment.

Goal: demonstrate event aggregation, a basic repeated-failed-login detection rule, and analyst recommendations.

## 1. Load synthetic events
These events are fabricated for demonstration and do not come from a real endpoint or Wazuh server.

In [ ]:
from collections import Counter
import json
from pathlib import Path

events = [
    {"timestamp":"2026-10-25T09:12:00","host":"WIN-TEST-01","event_type":"failed_login","source_ip":"192.0.2.15","severity":"medium"},
    {"timestamp":"2026-10-25T09:12:08","host":"WIN-TEST-01","event_type":"failed_login","source_ip":"192.0.2.15","severity":"medium"},
    {"timestamp":"2026-10-25T09:12:17","host":"WIN-TEST-01","event_type":"failed_login","source_ip":"192.0.2.15","severity":"medium"},
    {"timestamp":"2026-10-25T09:14:40","host":"UBUNTU-TEST-01","event_type":"successful_login","source_ip":"192.0.2.40","severity":"low"},
    {"timestamp":"2026-10-25T09:21:11","host":"UBUNTU-TEST-01","event_type":"failed_login","source_ip":"192.0.2.77","severity":"medium"},
    {"timestamp":"2026-10-25T09:21:19","host":"UBUNTU-TEST-01","event_type":"failed_login","source_ip":"192.0.2.77","severity":"medium"},
    {"timestamp":"2026-10-25T09:21:31","host":"UBUNTU-TEST-01","event_type":"failed_login","source_ip":"192.0.2.77","severity":"medium"},
]
print(f"Loaded {len(events)} synthetic events")
print("Event types:", dict(Counter(e["event_type"] for e in events)))

## 2. Apply a simple detection rule
Rule `DEMO-001` flags a source IP with three or more failed logins. This is illustrative logic, not a Wazuh rule.

In [ ]:
failed = [e for e in events if e["event_type"] == "failed_login"]
counts = Counter(e["source_ip"] for e in failed)
alerts = [
    {"rule":"DEMO-001", "source_ip":ip, "failed_attempts":count,
     "severity":"medium", "recommendation":"Review authentication logs and validate whether activity is authorized."}
    for ip, count in counts.items() if count >= 3
]
alerts

## 3. Analyst interpretation
- Repeated failed logins can indicate a mistyped password, a misconfigured service, or possible password guessing.
- An alert is not proof of compromise. Validate the source, account, time window, and surrounding events.
- In a live lab, the next step would be to verify this logic against actual Wazuh alerts and document evidence.

In [ ]:
Path("demo_events.json").write_text(
    json.dumps({"notice":"Synthetic educational data; not live telemetry.",
                "events":events, "alerts":alerts}, indent=2),
    encoding="utf-8"
)
print("Saved demo_events.json. Reminder: these are synthetic events, not live Wazuh telemetry.")

## 4. Limitations
This notebook does not install or communicate with Wazuh. To complete a real SIEM deployment, install the Wazuh all-in-one server, enroll an endpoint agent, collect real lab logs, and validate detection rules.